# 自定义工具

通过继承 `AgentTool` 并注册到 Agent，即可让 LLM 自动调用你的代码。

本 Notebook 实现一个 `add` 工具：计算两个数字之和。


In [1]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"


In [2]:
from typing import Any, Optional
from nova_agent import Agent, AgentTool, AgentToolResult
from nova_ai import TextContent

class AddTool(AgentTool[dict, dict]):
    name: str = "add"
    description: str = "计算两个数字之和"
    parameters: dict = {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "第一个数"},
            "b": {"type": "number", "description": "第二个数"},
        },
        "required": ["a", "b"],
    }
    label: str = "加法"

    async def execute(
        self,
        tool_call_id: str,
        params: Any,
        signal: Optional[Any] = None,
        on_update: Optional[Any] = None,
    ) -> AgentToolResult[dict]:
        a = params.get("a", 0)
        b = params.get("b", 0)
        result = a + b
        return AgentToolResult(
            content=[TextContent(text=f"{a} + {b} = {result}")],
            details={"a": a, "b": b, "result": result},
        )

agent = Agent(
    initial_state={"system_prompt": "你会在需要时调用 add 工具进行加法计算。"},
)
agent.set_tools([AddTool()])

agent.subscribe(lambda e: print(f"[{e.type}]" if e.type != "message_update" else ".", end=""))

await agent.prompt("请计算 23 + 45。")
await agent.wait_for_idle()

print("\n\n最终回复:", agent.state.messages[-1].content[0].text)


[agent_start][turn_start][message_start][message_end][message_start]....................[message_end][tool_execution_start][tool_execution_end][message_start][message_end][turn_end][turn_start][message_start].........[message_end][turn_end][agent_end]

最终回复: 23 + 45 = 68


## 观察工具执行

如果只想观察工具相关的 event，可以过滤 `tool_execution_start` / `tool_execution_end`。


In [3]:
def listen_tools(agent):
    def handler(event):
        if event.type == "tool_execution_start":
            print(f"开始执行: {event.tool_name}({event.args})")
        elif event.type == "tool_execution_end":
            print(f"结束执行: {event.tool_name} -> {event.result.content[0].text}")
    unsubscribe = agent.subscribe(handler)
    return unsubscribe

unsub = listen_tools(agent)
await agent.prompt("现在计算 100 + 200。")
await agent.wait_for_idle()
unsub()


[agent_start][turn_start][message_start][message_end][message_start]....................[message_end]开始执行: add({'a': 100, 'b': 200})
[tool_execution_start]结束执行: add -> 100 + 200 = 300
[tool_execution_end][message_start][message_end][turn_end][turn_start][message_start].........[message_end][turn_end][agent_end]